In [1]:
import math
import random
import torch
import torch.nn as nn
from torch.nn import functional as F
try:
    import torch_directml
    device = torch_directml.device()
    BACKEND = "DirectML"
except (ImportError, RuntimeError, Exception) as e:
    if torch.cuda.is_available():
        device = torch.device("cuda")
        BACKEND = "CUDA"
    else:
        device = torch.device("cpu")
        BACKEND = "CPU"

print(f"Backend: {BACKEND}")
print(f"Device: {device}")
batch_size = 16
block_size = 128
max_iters = 10000
learning_rate = 3e-4
eval_iters = 20
n_embd = 128
n_head = 4
n_layer = 3
dropout = 0.1
assert n_embd % n_head == 0

Backend: DirectML
Device: privateuseone:0


In [2]:
with open("train.txt", "r", encoding="utf-8") as f:
    text = f.read()
text = text[:500_000]
chars = sorted(set(text))
vocab_size = len(chars)
string_to_int = {ch: i for i, ch in enumerate(chars)}
int_to_string = {i: ch for i, ch in enumerate(chars)}
def encode(s):
    return [string_to_int[ch] for ch in s if ch in string_to_int]

def decode(ids):
    return "".join(int_to_string[int(i)] for i in ids)
data = torch.tensor(encode(text), dtype=torch.long)
if len(data) <= block_size + 1:
    raise ValueError("wizard_of_oz.txt is too short for the selected block_size.")
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]
print(f"Characters: {len(text):,}")
print(f"Vocabulary size: {vocab_size}")
print(f"Train tokens: {len(train_data):,} | Validation tokens: {len(val_data):,}")
def get_batch(split):
    source = train_data if split == "train" else val_data
    ix = torch.randint(len(source) - block_size, (batch_size,))
    x = torch.stack([source[i:i + block_size] for i in ix])
    y = torch.stack([source[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

Characters: 500,000
Vocabulary size: 83
Train tokens: 450,000 | Validation tokens: 50,000


In [3]:
class CausalSelfAttention(nn.Module):
    """Manual attention implementation for better DirectML compatibility."""
    def __init__(self, n_embd, n_head):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head
        self.head_dim = n_embd // n_head

        self.c_attn = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.c_proj = nn.Linear(n_embd, n_embd, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        # Boolean lower-triangular causal mask; moved with the model.
        mask = torch.tril(torch.ones(block_size, block_size, dtype=torch.bool))
        self.register_buffer("causal_mask", mask.view(1, 1, block_size, block_size))

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.c_attn(x).split(C, dim=2)

        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        # Scaled dot-product attention, written from basic tensor operations.
        scores = (q @ k.transpose(-2, -1)) * (self.head_dim ** -0.5)
        scores = scores.masked_fill(~self.causal_mask[:, :, :T, :T], float("-inf"))
        weights = F.softmax(scores, dim=-1)
        weights = self.attn_dropout(weights)

        out = weights @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(out))


class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ffwd = FeedForward(n_embd)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x


class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)

        # Tie input and output embeddings.
        self.lm_head.weight = self.token_embedding_table.weight
        self.apply(self._init_weights)

        for name, param in self.named_parameters():
            if name.endswith("c_proj.weight"):
                nn.init.normal_(param, mean=0.0, std=0.02 / math.sqrt(2 * n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, index, targets=None):
        B, T = index.shape
        if T > block_size:
            raise ValueError(f"Sequence length {T} exceeds block_size={block_size}.")

        positions = torch.arange(T, device=index.device)
        x = self.token_embedding_table(index) + self.position_embedding_table(positions)
        x = self.blocks(x)
        logits = self.lm_head(self.ln_f(x))

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(B * T, -1), targets.reshape(B * T))
        return logits, loss

    @torch.no_grad()
    def generate(self, index, max_new_tokens, temperature=0.8, top_k=40):
        self.eval()
        for _ in range(max_new_tokens):
            index_cond = index[:, -block_size:]
            logits, _ = self(index_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-6)

            if top_k is not None:
                k = min(top_k, logits.size(-1))
                values, _ = torch.topk(logits, k)
                logits[logits < values[:, [-1]]] = float("-inf")

            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            index = torch.cat((index, next_token), dim=1)
        return index


model = GPTLanguageModel(vocab_size).to(device)
num_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {num_params:,}")

Model parameters: 631,168


In [4]:
@torch.no_grad()
def estimate_loss():
    model.eval()
    results = {}
    for split in ("train", "val"):
        losses = []
        for _ in range(eval_iters):
            xb, yb = get_batch(split)
            _, loss = model(xb, yb)
            losses.append(loss.detach().cpu().item())
        results[split] = sum(losses) / len(losses)
    model.train()
    return results

In [5]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate,
    betas=(0.9, 0.95),
    weight_decay=0.1,
)

def get_lr(step):
    warmup_iters = 50
    min_lr = learning_rate / 10
    if step < warmup_iters:
        return learning_rate * (step + 1) / warmup_iters
    decay_ratio = min(1.0, (step - warmup_iters) / max(1, max_iters - warmup_iters))
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return min_lr + coeff * (learning_rate - min_lr)

model.train()
for step in range(max_iters):
    lr = get_lr(step)
    for group in optimizer.param_groups:
        group["lr"] = lr

    if step % 50 == 0:
        losses = estimate_loss()
        print(
            f"step {step:4d} | "
            f"train: {losses['train']:.4f} | "
            f"val: {losses['val']:.4f} | lr: {lr:.6g}"
        )

    xb, yb = get_batch("train")
    _, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

print(f"Training finished. Last batch loss: {loss.detach().cpu().item():.4f}")

# Save portable CPU weights instead of pickling the entire model object.
checkpoint = {
    "model_state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()},
    "chars": chars,
    "config": {
        "block_size": block_size, "n_embd": n_embd,
        "n_head": n_head, "n_layer": n_layer, "dropout": dropout,
    },
}
torch.save(checkpoint, "fcc.pt")
print("Saved: fcc.pt")

step    0 | train: 4.4233 | val: 4.4231 | lr: 6e-06


C:\Users\DELL\anaconda3\envs\py310\lib\site-packages\torch\optim\adamw.py:529: UserWarning: The operator 'aten::lerp.Scalar_out' is not currently supported on the DML backend and will fall back to run on the CPU. This may have performance implications. (Triggered internally at C:\__w\1\s\pytorch-directml-plugin\torch_directml\csrc\dml\dml_cpu_fallback.cpp:17.)
  torch._foreach_lerp_(device_exp_avgs, device_grads, 1 - beta1)


step   50 | train: 3.1308 | val: 3.1361 | lr: 0.0003
step  100 | train: 2.6707 | val: 2.6601 | lr: 0.000299983
step  150 | train: 2.5577 | val: 2.5572 | lr: 0.000299933
step  200 | train: 2.5072 | val: 2.5063 | lr: 0.000299849
step  250 | train: 2.4540 | val: 2.4836 | lr: 0.000299731
step  300 | train: 2.4330 | val: 2.4514 | lr: 0.00029958
step  350 | train: 2.4293 | val: 2.4419 | lr: 0.000299395
step  400 | train: 2.3912 | val: 2.4140 | lr: 0.000299177
step  450 | train: 2.3805 | val: 2.4034 | lr: 0.000298925
step  500 | train: 2.3696 | val: 2.3810 | lr: 0.00029864
step  550 | train: 2.3524 | val: 2.3667 | lr: 0.000298321
step  600 | train: 2.3363 | val: 2.3422 | lr: 0.00029797
step  650 | train: 2.3173 | val: 2.3326 | lr: 0.000297585
step  700 | train: 2.3114 | val: 2.3397 | lr: 0.000297167
step  750 | train: 2.2744 | val: 2.3086 | lr: 0.000296716
step  800 | train: 2.2591 | val: 2.2741 | lr: 0.000296233
step  850 | train: 2.2400 | val: 2.2621 | lr: 0.000295716
step  900 | train: 2.2

In [8]:
# Generate a sample.
prompt = "bade the voyagers"
if not all(ch in string_to_int for ch in prompt):
    prompt = ""

context = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
model.eval()
sample = model.generate(
    context,
    max_new_tokens=300,
    temperature=0.5,
    top_k=40,
)
print(decode(sample[0].detach().cpu().tolist()))

bade the voyagers of the brother than the world,

The time of my be subject stand of the world,

That shall thou shalt that honour she dispered,

Where thou art a conquertant his entreath,

And the live the content of the leaves still,

And beauty the honour that that have in my life,

Thou hast thy rance disted the
